In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   PIPELINE COMPLET : FINE-TUNING + TEST + ÉVALUATION        ║
# ║   Qwen2.5-3B-Instruct sur Google Colab                      ║
# ╚══════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════
# CELLULE 1 — INSTALLATION
# ══════════════════════════════════════════════════════════════
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q transformers accelerate torch pandas openpyxl \
    datasets peft trl "bitsandbytes>=0.46.1"

In [1]:
print('⏳ Mise à jour de torchao...')
!pip install --upgrade torchao
print('✅ torchao mis à jour')

⏳ Mise à jour de torchao...
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 2.5 MB/s eta 0:00:00
✅ torchao mis à jour


In [ ]:
 #══════════════════════════════════════════════════════════════
# CELLULE 2 — MONTAGE DRIVE + IMPORTS
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import gc, json, torch, time, warnings
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
warnings.filterwarnings('ignore')

print("✅ Imports OK")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


✅ Imports OK
GPU : Tesla T4
VRAM : 14.6 GB


In [ ]:
print('⏳ Réinstallation de pyarrow, datasets et pandas pour résoudre l\'erreur de compatibilité...')
!pip uninstall -y pyarrow datasets pandas
!pip install pyarrow datasets pandas==2.2.2
print('✅ Packages réinstallés. Veuillez relancer la cellule précédente si l\'erreur persiste.')

⏳ Réinstallation de pyarrow, datasets et pandas pour résoudre l'erreur de compatibilité...
Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (48.9 MB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)


✅ Packages réinstallés. Veuillez relancer la cellule précédente si l'erreur persiste.


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIGURATION
# ══════════════════════════════════════════════════════════════
import os

# ── Chemins ───────────────────────────────────────────────────
BASE               = "/content/drive/MyDrive/Lyne"
CHEMIN_TRAIN       = f"{BASE}/dataset/train1.jsonl"
CHEMIN_VAL         = f"{BASE}/dataset/validation1.jsonl"
CHEMIN_TEST_DAILY  = f"{BASE}/test-daily.xlsx"    # fichier à tester
CHEMIN_VALID_DAILY = f"{BASE}/valid-daily.xlsx"   # fichier avec causes validées
DOSSIER_LORA       = f"{BASE}/finetuned_lora"
DOSSIER_MERGE      = f"{BASE}/model_final"
RESULTATS_TEST     = f"{BASE}/resultats_test.xlsx"
RESULTATS_EVAL     = f"{BASE}/evaluation.xlsx"

os.makedirs(DOSSIER_LORA,  exist_ok=True)
os.makedirs(DOSSIER_MERGE, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# ── Paramètres LoRA ───────────────────────────────────────────
LORA_R       = 32       # plus grand = apprend mieux
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05
LORA_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]  # tous les modules

# ── Paramètres entraînement ───────────────────────────────────
NUM_EPOCHS     = 5      # plus d'epochs pour mieux apprendre
BATCH_SIZE     = 1
GRAD_ACCUM     = 8      # batch effectif = 8
LEARNING_RATE  = 1e-4   # plus faible = apprentissage plus stable
MAX_SEQ_LENGTH = 768    # compromis mémoire / qualité
WARMUP_RATIO   = 0.05
WEIGHT_DECAY   = 0.01

# ── 15 catégories officielles ─────────────────────────────────
CATEGORIES = {
    "MPR issue", "BSS Hardware issue",
    "AKTIVCO Defaut GE & Power Cabinet",
    "AKTIVCO Coupure ENEO & Baisse de tension",
    "Coupure ENEO & Baisse de tension",
    "Defaut GE & Power Cabinet",
    "Sharing", "SPARE-ISSUE", "Fiber AOF",
    "Projet OCM (HUAWEI)", "ACCESS-ISSUE",
    "Projet OCM (ZTE, NOKIA, autres projets)",
    "ODU HS", "IP & VLAN", "Warehouse HUAWEI",
}

print("✅ Configuration OK")

✅ Configuration OK


In [ ]:
#  ══════════════════════════════════════════════════════════════
# CELLULE 4 — CHARGEMENT DES DONNÉES
# ══════════════════════════════════════════════════════════════

def charger_jsonl(chemin):
    data = []
    with open(chemin, "r", encoding="utf-8") as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne:
                data.append(json.loads(ligne))
    return data

train_data = charger_jsonl(CHEMIN_TRAIN)
val_data   = charger_jsonl(CHEMIN_VAL)

print(f"Train      : {len(train_data)} exemples")
print(f"Validation : {len(val_data)} exemples")

# Vérifier la distribution des causes
from collections import Counter
causes_train = [ex["messages"][2]["content"].replace("Cause : ","")
                for ex in train_data]
dist = Counter(causes_train)
print("\n📊 Distribution train :")
for c, n in sorted(dist.items(), key=lambda x: x[1], reverse=True):
    statut = "✅" if n >= 30 else "⚠️ "
    print(f"  {statut} {c}: {n}")


Train      : 794 exemples
Validation : 99 exemples

📊 Distribution train :
  ✅ BSS Hardware issue: 63
  ✅ ACCESS-ISSUE: 61
  ✅ MPR issue: 61
  ✅ Fiber AOF: 60
  ✅ Projet OCM (HUAWEI): 57
  ✅ Projet OCM (ZTE, NOKIA, autres projets): 56
  ✅ Sharing: 55
  ✅ Defaut GE & Power Cabinet: 54
  ✅ EXCLU: 51
  ✅ SPARE-ISSUE: 51
  ✅ ODU HS: 51
  ✅ IP & VLAN: 48
  ✅ Warehouse HUAWEI: 39
  ✅ AKTIVCO Coupure ENEO & Baisse de tension: 36
  ✅ AKTIVCO Defaut GE & Power Cabinet: 35
  ⚠️  Coupure ENEO & Baisse de tension: 16


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 5 — CHARGEMENT DU MODÈLE
# ══════════════════════════════════════════════════════════════

gc.collect()
torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("⏳ Chargement tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print("⏳ Chargement modèle (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print(f"✅ Modèle chargé — VRAM : {torch.cuda.memory_allocated()/1024**3:.1f} GB")

⏳ Chargement tokenizer...


⏳ Chargement modèle (4-bit)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Modèle chargé — VRAM : 1.9 GB


In [ ]:
#  ══════════════════════════════════════════════════════════════
# CELLULE 6 — PRÉPARATION DU DATASET
# ══════════════════════════════════════════════════════════════

def formater(exemple):
    return {"text": tokenizer.apply_chat_template(
        exemple["messages"], tokenize=False, add_generation_prompt=False
    )}

def tokeniser(exemple):
    t = tokenizer(exemple["text"], truncation=True,
                  max_length=MAX_SEQ_LENGTH, padding=False)
    t["labels"] = t["input_ids"].copy()
    return t

train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

train_ds = train_ds.map(formater, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(formater,   remove_columns=val_ds.column_names)

train_tok = train_ds.map(tokeniser, remove_columns=["text"], batched=False)
val_tok   = val_ds.map(tokeniser,   remove_columns=["text"], batched=False)

steps_ep     = len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)
total_steps  = steps_ep * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

print(f"Steps/epoch : {steps_ep} | Total : {total_steps} | Warmup : {warmup_steps}")

Map:   0%|          | 0/794 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

Map:   0%|          | 0/794 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

Steps/epoch : 99 | Total : 495 | Warmup : 24


In [ ]:
#  ══════════════════════════════════════════════════════════════
# CELLULE 7 — CONFIGURATION LORA + TRAINER
# ══════════════════════════════════════════════════════════════

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=LORA_MODULES,
)
model = get_peft_model(model, lora_config)
print("✅ LoRA appliqué")
model.print_trainable_parameters()

sft_config = SFTConfig(
    output_dir=DOSSIER_LORA,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    optim="paged_adamw_8bit",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=0,
    group_by_length=True,
    max_grad_norm=1.0,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    args=sft_config,
)
print("✅ Trainer configuré")

✅ LoRA appliqué
trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031
✅ Trainer configuré


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 8 — ENTRAÎNEMENT
# ══════════════════════════════════════════════════════════════

print("\n🚀 LANCEMENT ENTRAÎNEMENT")
print("=" * 50)
trainer.train()
print("✅ ENTRAÎNEMENT TERMINÉ")

model.save_pretrained(DOSSIER_LORA)
tokenizer.save_pretrained(DOSSIER_LORA)
print(f"✅ Adaptateurs LoRA sauvegardés : {DOSSIER_LORA}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



🚀 LANCEMENT ENTRAÎNEMENT


Step,Training Loss,Validation Loss
50,0.375219,0.451324
100,0.186701,0.353765
150,0.235684,0.304251
200,0.120149,0.279795
250,0.206070,0.258458
300,0.099307,0.241521
350,0.128905,0.245191
400,0.076141,0.230792
450,0.110242,0.239096
500,0.094210,0.238547


✅ ENTRAÎNEMENT TERMINÉ
✅ Adaptateurs LoRA sauvegardés : /content/drive/MyDrive/Lyne/finetuned_lora


In [ ]:
# Télécharger directement via l'API Google Drive (contourne le FUSE)
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError
import io, os

auth.authenticate_user()
service = build('drive', 'v3')

# Lister les fichiers dans le dossier
DOSSIER_LOCAL = "/content/finetuned_lora_local"
os.makedirs(DOSSIER_LOCAL, exist_ok=True)

# Trouver l'ID du dossier finetuned_lora
try:
    results = service.files().list(
        q="name='finetuned_lora' and mimeType='application/vnd.google-apps.folder'",
        fields="files(id, name)"
    ).execute()
    folder_id = results['files'][0]['id']
    print(f"✅ Dossier trouvé, ID: {folder_id}")
except IndexError:
    print(f"❌ Dossier 'finetuned_lora' introuvable dans Google Drive.")
    folder_id = None


if folder_id:
    # Lister et télécharger tous les fichiers
    files = service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, size, mimeType)" # Request mimeType
    ).execute()['files']

    print(f"📁 {len(files)} fichiers trouvés :")
    for f in files:
        print(f"  - {f['name']} (MIME: {f.get('mimeType', 'N/A')}, Size: {f.get('size', '?')} bytes)")

        if f.get('mimeType') == 'application/vnd.google-apps.folder':
            print("    ⚠️ C'est un dossier, skipping download.")
            continue

        request = service.files().get_media(fileId=f['id'])
        path = os.path.join(DOSSIER_LOCAL, f['name'])
        try:
            with open(path, 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, request)
                done = False
                while not done:
                    _, done = downloader.next_chunk()
            print(f"    ✅ Téléchargé")
        except HttpError as e:
            if e.resp.status == 403 and "fileNotDownloadable" in str(e):
                print(f"    ❌ Impossible de télécharger '{f['name']}'. Il s'agit peut-être d'un fichier Google Workspace ou d'un alias de dossier qui ne peut pas être téléchargé directement. Erreur: {e}")
            else:
                print(f"    ❌ Erreur lors du téléchargement de '{f['name']}': {e}")
        except Exception as e:
            print(f"    ❌ Une erreur inattendue s'est produite lors du téléchargement de '{f['name']}': {e}")

    print("\n✅ Tous les fichiers locaux :", os.listdir(DOSSIER_LOCAL))
else:
    print("Aucun dossier 'finetuned_lora' trouvé ou spécifié, skipping file download.")

✅ Dossier trouvé, ID: 1PQpB6NyuH4m651_SzhbXC6MLvzIbz54b
📁 8 fichiers trouvés :
  - README.md (MIME: text/markdown, Size: 1481 bytes)
    ✅ Téléchargé
  - tokenizer.json (MIME: application/json, Size: 11421990 bytes)
    ✅ Téléchargé
  - tokenizer_config.json (MIME: application/json, Size: 662 bytes)
    ✅ Téléchargé
  - chat_template.jinja (MIME: application/octet-stream, Size: 2507 bytes)
    ✅ Téléchargé
  - adapter_config.json (MIME: application/json, Size: 1103 bytes)
    ✅ Téléchargé
  - adapter_model.safetensors (MIME: application/octet-stream, Size: 239536272 bytes)
    ✅ Téléchargé
  - checkpoint-500 (MIME: application/vnd.google-apps.folder, Size: ? bytes)
    ⚠️ C'est un dossier, skipping download.
  - checkpoint-400 (MIME: application/vnd.google-apps.folder, Size: ? bytes)
    ⚠️ C'est un dossier, skipping download.

✅ Tous les fichiers locaux : ['README.md', 'checkpoint-500', 'tokenizer_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'ada

In [ ]:
import os

DOSSIER_LOCAL = "/content/finetuned_lora_local"
print("Fichiers présents :", os.listdir(DOSSIER_LOCAL))

Fichiers présents : []


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 9 — MERGE LORA + MODÈLE DE BASE
# ══════════════════════════════════════════════════════════════
import gc, torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE          = "/content/drive/MyDrive/Lyne"
DOSSIER_LORA  = f"{BASE}/finetuned_lora"
DOSSIER_MERGE = f"{BASE}/model_final"
MODEL_NAME    = "Qwen/Qwen2.5-3B-Instruct"

gc.collect()
torch.cuda.empty_cache()

print("⏳ Chargement modèle de base (GPU float16)...")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",       # ← GPU au lieu de CPU
    trust_remote_code=True,
)

print("⏳ Merge des adaptateurs LoRA...")
model_merge = PeftModel.from_pretrained(model_base, DOSSIER_LORA)
model_merge = model_merge.merge_and_unload()

# Déplacer sur CPU pour la sauvegarde → libère la VRAM
model_merge = model_merge.to("cpu")

print("⏳ Sauvegarde modèle final...")
os.makedirs(DOSSIER_MERGE, exist_ok=True)
model_merge.save_pretrained(
    DOSSIER_MERGE,
    max_shard_size="2GB",
    safe_serialization=True
)

tokenizer = AutoTokenizer.from_pretrained(DOSSIER_LORA, trust_remote_code=True)
tokenizer.save_pretrained(DOSSIER_MERGE)

del model_merge, model_base
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ Modèle final sauvegardé : {DOSSIER_MERGE}")
for f in sorted(os.listdir(DOSSIER_MERGE)):
    taille = os.path.getsize(f"{DOSSIER_MERGE}/{f}") / 1024**2
    print(f"   {f:<40} {taille:.1f} MB")

⏳ Chargement modèle de base (GPU float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

⏳ Merge des adaptateurs LoRA...
⏳ Sauvegarde modèle final...


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]


✅ Modèle final sauvegardé : /content/drive/MyDrive/Lyne/model_final
   chat_template.jinja                      0.0 MB
   config.json                              0.0 MB
   generation_config.json                   0.0 MB
   model-00001-of-00004.safetensors         1899.6 MB
   model-00002-of-00004.safetensors         1867.2 MB
   model-00003-of-00004.safetensors         1868.2 MB
   model-00004-of-00004.safetensors         251.0 MB
   model.safetensors.index.json             0.0 MB
   tokenizer.json                           10.9 MB
   tokenizer_config.json                    0.0 MB


In [ ]:
#  ══════════════════════════════════════════════════════════════
# CELLULE 10 — FONCTION D'INFÉRENCE
# ══════════════════════════════════════════════════════════════


# Liste officielle des 15 catégories — utilisée pour la normalisation
CATEGORIES = [
    "MPR issue",
    "BSS Hardware issue",
    "AKTIVCO Defaut GE & Power Cabinet",
    "AKTIVCO Coupure ENEO & Baisse de tension",
    "Coupure ENEO & Baisse de tension",
    "Defaut GE & Power Cabinet",
    "Sharing",
    "SPARE-ISSUE",
    "Fiber AOF",
    "Projet OCM (HUAWEI)",
    "ACCESS-ISSUE",
    "Projet OCM (ZTE, NOKIA, autres projets)",
    "ODU HS",
    "IP & VLAN",
    "Warehouse HUAWEI",
]
print(f"✅ {len(CATEGORIES)} catégories définies")
# System prompt identique à l'entraînement
SYSTEM_PROMPT = """Tu es un expert en analyse d'incidents réseau télécom au monde.

Tu reçois un commentaire d'incident. Selon le type d'incident, tu peux aussi recevoir :
- Owner : le gestionnaire du site
- Topology : la configuration électrique du site,l'entreprise qui s'occupe de l'energie sur le site
- Vendors : l'équipementier radio (ZTE, HUAWEI, NOKIA)
je veux que tu analyse le commentaire , que tu le comprenne , que tu raisonne que tu comprenne si l'incident est cloturer ou si il est encore en cours.
Tu dois identifier la cause racine parmi ces 15 catégories EXACTES :
1. MPR issue
2. BSS Hardware issue
3. AKTIVCO Defaut GE & Power Cabinet
4. AKTIVCO Coupure ENEO & Baisse de tension
5. Coupure ENEO & Baisse de tension
6. Defaut GE & Power Cabinet
7. Sharing
8. SPARE-ISSUE
9. Fiber AOF
10. Projet OCM (HUAWEI)
11. ACCESS-ISSUE
12. Projet OCM (ZTE, NOKIA, autres projets)
13. ODU HS
14. IP & VLAN
15. Warehouse HUAWEI

Réponds UNIQUEMENT avec : Cause : [la cause exacte de la liste ci-dessus]"""

BESOIN_OT = {
    "AKTIVCO Defaut GE & Power Cabinet",
    "AKTIVCO Coupure ENEO & Baisse de tension",
    "Defaut GE & Power Cabinet",
    "Coupure ENEO & Baisse de tension",
    "Sharing",
}
BESOIN_V = {
    "Projet OCM (HUAWEI)",
    "Projet OCM (ZTE, NOKIA, autres projets)",
}

def propre(v):
    s = str(v).strip()
    return "" if s.lower() in ["nan","n/a","none","","-"] else s

def construire_prompt(commentaire, owner="", topology="", vendors=""):
    user = f"Commentaire : {commentaire.strip()}"
    if propre(owner):    user += f"\nOwner : {propre(owner)}"
    if propre(topology): user += f"\nTopology : {propre(topology)}"
    if propre(vendors):  user += f"\nVendors : {propre(vendors)}"
    return user

def predire(commentaire, owner="", topology="", vendors="",
            model_obj=None, tok=None):
    """
    Retourne (analyse, cause_predite, justification).
    - analyse      : ce que le modèle comprend du commentaire
    - cause_predite: la cause officielle extraite
    - justification: la cause brute retournée par le modèle
    """
    user_content = construire_prompt(commentaire, owner, topology, vendors)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(prompt, return_tensors="pt").to(model_obj.device)

    with torch.no_grad():
        out = model_obj.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )
    reponse = tok.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Extraire la cause
    cause = ""
    if "Cause :" in reponse:
        cause = reponse.split("Cause :")[-1].strip().split("\n")[0].strip()
    elif "Cause:" in reponse:
        cause = reponse.split("Cause:")[-1].strip().split("\n")[0].strip()
    else:
        cause = reponse[:100].strip()

    # Normaliser vers une catégorie officielle
    cause_norm = cause
    for cat in CATEGORIES:
        if cat.lower() in cause.lower():
            cause_norm = cat
            break

    # Analyse = résumé du commentaire (les 200 premiers caractères pertinents)
    analyse = commentaire[:200].replace("\n", " ").strip()
    if len(commentaire) > 200:
        analyse += "..."

    return analyse, cause_norm, cause

✅ 15 catégories définies


In [ ]:
# # ══════════════════════════════════════════════════════════════
# CELLULE 11 — CHARGEMENT MODÈLE FINE-TUNÉ POUR INFÉRENCE
# ══════════════════════════════════════════════════════════════

import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Redefine configuration variables for standalone execution
BASE               = "/content/drive/MyDrive/Lyne"
DOSSIER_LORA       = f"{BASE}/finetuned_lora"
DOSSIER_MERGE      = f"{BASE}/model_final"
MODEL_NAME         = "Qwen/Qwen2.5-3B-Instruct"

gc.collect()
torch.cuda.empty_cache()

print("⏳ Chargement modèle fine-tuné pour inférence...")

# Option A : charger depuis le merge (recommandé)
try:
    from transformers import pipeline as hf_pipeline
    infer_tokenizer = AutoTokenizer.from_pretrained(
        DOSSIER_MERGE, trust_remote_code=True
    )
    infer_tokenizer.pad_token    = infer_tokenizer.eos_token
    infer_tokenizer.padding_side = "right"

    infer_model = AutoModelForCausalLM.from_pretrained(
        DOSSIER_MERGE,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("✅ Modèle mergé chargé")

# Option B : charger base + LoRA si le merge n'existe pas encore
except Exception as e:
    print(f"⚠️ Merge non trouvé ({e}), chargement base + LoRA...")
    infer_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, trust_remote_code=True
    )
    infer_tokenizer.pad_token    = infer_tokenizer.eos_token
    infer_tokenizer.padding_side = "right"

    quant2 = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=quant2,
        device_map="auto", trust_remote_code=True,
    )
    infer_model = PeftModel.from_pretrained(base, DOSSIER_LORA)
    infer_model.eval()
    print("✅ Modèle base + LoRA chargé")

infer_model.eval()

⏳ Chargement modèle fine-tuné pour inférence...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Modèle mergé chargé


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE UNIQUE — TEST ET ÉVALUATION COMPLÈTE
# Fichier test    : test2.xlsx        (2500 lignes)
# Fichier valid   : validation2.xlsx  (CAUSE + Verification + Codesite)
# ══════════════════════════════════════════════════════════════

import os, re, time, warnings
import pandas as pd
from collections import Counter, defaultdict
from google.colab import drive, files

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# ÉTAPE 0 — MONTAGE DRIVE ET CONFIGURATION
# ─────────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)

BASE           = "/content/drive/MyDrive/Lyne"
CHEMIN_TEST    = f"{BASE}/test2.xlsx"
CHEMIN_VALID   = f"{BASE}/validation2.xlsx"
RESULTATS_TEST = f"{BASE}/resultats_test2.xlsx"
RAPPORT_PERF   = f"{BASE}/rapport_performance.xlsx"

# Catégories sur lesquelles le modèle a été entraîné
CATEGORIES_ENTRAINEMENT = [
    "MPR issue",
    "BSS Hardware issue",
    "AKTIVCO Defaut GE & Power Cabinet",
    "AKTIVCO Coupure ENEO & Baisse de tension",
    "Coupure ENEO & Baisse de tension",
    "Defaut GE & Power Cabinet",
    "Sharing",
    "SPARE-ISSUE",
    "Fiber AOF",
    "Projet OCM (HUAWEI)",
    "ACCESS-ISSUE",
    "Projet OCM (ZTE, NOKIA, autres projets)",
    "ODU HS",
    "IP & VLAN",
    "Warehouse HUAWEI",
]

# Mots-clés pour détecter les sites impactés
MOTS_CLES_IMPACT = [
    r"indisponibilit[eé] des services? voix\s*[&et]+\s*data dans la localit[eé] de",
    r"impacted?\s+by",
    r"impact[eé]r?\s+par",
    r"incident\s+at",
    r"probabl[e]?\s+probl[eè]me\s+de\s+.{0,30}\s+sur le site\s+de",
    r"affect(?:ed)?\s+by",
]
PATTERN_IMPACT = re.compile(
    "|".join(MOTS_CLES_IMPACT),
    re.IGNORECASE
)

# Pattern pour extraire code site (ex: LIT_123, CTR_456, SUO_789)
PATTERN_CODE_SITE = re.compile(
    r'\b([A-Z]{2,4}_\d{3,4})\b',
    re.IGNORECASE
)

print("="*60)
print("✅ Configuration OK")
print(f"   Test    : {CHEMIN_TEST}")
print(f"   Valid   : {CHEMIN_VALID}")
print(f"   Sortie  : {RESULTATS_TEST}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 1 — FONCTIONS UTILITAIRES
# ─────────────────────────────────────────────────────────────

def propre(v):
    s = str(v).strip()
    return "" if s.lower() in ["nan", "n/a", "none", "", "-"] else s

def est_impacte(commentaire):
    """Retourne True si le commentaire indique un site impacté."""
    return bool(PATTERN_IMPACT.search(commentaire))

def extraire_site_impactant(commentaire):
    """
    Extrait le code site du site qui impacte.
    Cherche d'abord un code site (ex: LIT_123) dans le commentaire.
    """
    codes = PATTERN_CODE_SITE.findall(commentaire)
    if codes:
        return codes[0].upper()
    return None

def construire_prompt_test(commentaire, owner="", topology="", vendors=""):
    """Construit le prompt USER pour l'inférence."""
    user = f"Commentaire : {commentaire.strip()}"
    if propre(owner):    user += f"\nOwner : {propre(owner)}"
    if propre(topology): user += f"\nTopology : {propre(topology)}"
    if propre(vendors):  user += f"\nVendors : {propre(vendors)}"
    return user

# ─────────────────────────────────────────────────────────────
# ÉTAPE 2 — LECTURE DU FICHIER TEST
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📂 LECTURE DU FICHIER TEST")
print("="*60)

df = pd.read_excel(CHEMIN_TEST, dtype=str)
print(f"   Lignes totales    : {len(df)}")
print(f"   Colonnes          : {list(df.columns)}")

# Vérifier colonnes essentielles
assert "Verifications" in df.columns, "❌ Colonne 'Verifications' introuvable"
assert "Codesite"      in df.columns, "❌ Colonne 'Codesite' introuvable"

# Colonnes contextuelles optionnelles
col_own = next((c for c in df.columns if c.lower() == "owner"), None)
col_top = next((c for c in df.columns
                if "topology" in c.lower() or "topolopgy" in c.lower()), None)
col_ven = next((c for c in df.columns if "vendor" in c.lower()), None)

# Initialiser les colonnes de sortie
df["TYPE_CAS"]      = ""   # NORMAL / IMPACTE / NO_ROOT_CAUSE
df["ANALYSE"]       = ""   # Analyse du commentaire par le modèle
df["JUSTIFICATION"] = ""   # Justification de la prédiction
df["CAUSE_PREDITE"] = ""   # Cause prédite
df["SITE_IMPACTANT"]= ""   # Pour les cas impactés : code du site impactant
df["TEMPS_S"]       = 0.0  # Temps de traitement en secondes

# ─────────────────────────────────────────────────────────────
# ÉTAPE 3 — CLASSIFICATION DES LIGNES
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🔍 CLASSIFICATION DES LIGNES")
print("="*60)

idx_no_root    = []  # Pas de commentaire
idx_impactes   = []  # Sites impactés
idx_normaux    = []  # Cas normaux à traiter par le modèle

for idx, row in df.iterrows():
    com = propre(str(row.get("Verifications", "")))

    if not com:
        df.at[idx, "TYPE_CAS"]      = "NO_ROOT_CAUSE"
        df.at[idx, "CAUSE_PREDITE"] = "no root cause"
        df.at[idx, "ANALYSE"]       = "Aucun commentaire disponible pour ce site."
        df.at[idx, "JUSTIFICATION"] = "Pas d'information sur l'incident."
        idx_no_root.append(idx)
    elif est_impacte(com):
        df.at[idx, "TYPE_CAS"] = "IMPACTE"
        idx_impactes.append(idx)
    else:
        df.at[idx, "TYPE_CAS"] = "NORMAL"
        idx_normaux.append(idx)

print(f"   No root cause : {len(idx_no_root)}")
print(f"   Cas normaux   : {len(idx_normaux)}")
print(f"   Cas impactés  : {len(idx_impactes)}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 4 — TRAITEMENT 1 : CAS NORMAUX (modèle fine-tuné)
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🤖 TRAITEMENT 1 — CAS NORMAUX")
print("="*60)

traites   = 0
t_debut   = time.time()

# Dictionnaire codesite → cause pour résoudre les cas impactés ensuite
dict_causes_par_site = {}

for i, idx in enumerate(idx_normaux):
    row = df.loc[idx]
    com = propre(str(row.get("Verifications", "")))
    own = propre(str(row.get(col_own, ""))) if col_own else ""
    top = propre(str(row.get(col_top, ""))) if col_top else ""
    ven = propre(str(row.get(col_ven, ""))) if col_ven else ""

    t0 = time.time()
    try:
        _, cause_norm, cause_brute = predire(
            com, own, top, ven, infer_model, infer_tokenizer
        )
        # Analyse : résumé du commentaire
        analyse = com[:250].replace("\n", " ").strip()
        if len(com) > 250:
            analyse += "..."
        justification = cause_brute

    except Exception as e:
        cause_norm    = "ERREUR"
        cause_brute   = str(e)[:80]
        analyse       = com[:150]
        justification = f"Erreur modèle : {str(e)[:80]}"

    df.at[idx, "CAUSE_PREDITE"] = cause_norm
    df.at[idx, "ANALYSE"]       = analyse
    df.at[idx, "JUSTIFICATION"] = justification
    df.at[idx, "TEMPS_S"]       = round(time.time() - t0, 2)

    # Enregistrer dans le dictionnaire pour les cas impactés
    codesite = propre(str(row.get("Codesite", ""))).upper()
    if codesite and cause_norm not in ["ERREUR", ""]:
        dict_causes_par_site[codesite] = {
            "cause"        : cause_norm,
            "justification": justification,
            "analyse"      : analyse,
        }

    traites += 1
    if traites % 50 == 0 or traites == 1:
        elapsed = time.time() - t_debut
        vitesse = traites / elapsed if elapsed > 0 else 0
        restant = (len(idx_normaux) - traites) / vitesse if vitesse > 0 else 0
        print(f"   {traites}/{len(idx_normaux)} | {vitesse:.2f} ligne/s | ETA {restant/60:.1f} min")

print(f"   ✅ {len(idx_normaux)} cas normaux traités")
print(f"   📚 {len(dict_causes_par_site)} sites indexés pour résolution des cas impactés")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 5 — TRAITEMENT 2 : CAS IMPACTÉS
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🔗 TRAITEMENT 2 — CAS IMPACTÉS")
print("="*60)

resolus   = 0
non_resolus = 0

for idx in idx_impactes:
    row = df.loc[idx]
    com = propre(str(row.get("Verifications", "")))

    # Extraire le code site impactant depuis le commentaire
    code_impactant = extraire_site_impactant(com)
    df.at[idx, "SITE_IMPACTANT"] = code_impactant or "non trouvé"

    if code_impactant and code_impactant.upper() in dict_causes_par_site:
        # Résolution par code site exact
        info = dict_causes_par_site[code_impactant.upper()]
        df.at[idx, "CAUSE_PREDITE"] = info["cause"]
        df.at[idx, "ANALYSE"]       = (
            f"Site impacté par {code_impactant}. "
            f"Cause héritée du site impactant. "
            f"Analyse du site source : {info['analyse'][:150]}"
        )
        df.at[idx, "JUSTIFICATION"] = (
            f"Ce site est impacté par {code_impactant} "
            f"dont la cause est : {info['cause']}"
        )
        resolus += 1

    else:
        # Site impactant non trouvé dans les cas normaux traités
        df.at[idx, "CAUSE_PREDITE"] = "SITE_IMPACTANT_NON_RESOLU"
        df.at[idx, "ANALYSE"]       = (
            f"Site impacté détecté. "
            f"Site impactant identifié : {code_impactant or 'non trouvé dans le commentaire'}. "
            f"Impossible de résoudre car le site source n'est pas dans le fichier."
        )
        df.at[idx, "JUSTIFICATION"] = (
            f"Site impactant ({code_impactant or 'inconnu'}) "
            f"absent du fichier de traitement."
        )
        non_resolus += 1

print(f"   ✅ Cas impactés résolus     : {resolus}")
print(f"   ⚠️  Cas impactés non résolus : {non_resolus}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 6 — SAUVEGARDE DU FICHIER TEST ENRICHI
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("💾 SAUVEGARDE FICHIER TEST ENRICHI")
print("="*60)

# Réorganiser les colonnes : colonnes importantes en premier
cols_prio  = ["Codesite", "Verifications", "TYPE_CAS", "SITE_IMPACTANT",
              "ANALYSE", "JUSTIFICATION", "CAUSE_PREDITE", "TEMPS_S"]
cols_reste = [c for c in df.columns if c not in cols_prio]
cols_final = [c for c in cols_prio if c in df.columns] + cols_reste
df[cols_final].to_excel(RESULTATS_TEST, index=False, engine='openpyxl')

print(f"   ✅ Sauvegardé : {RESULTATS_TEST}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 7 — ÉVALUATION AVEC validation2.xlsx
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📊 ÉVALUATION AVEC validation2.xlsx")
print("="*60)

df_valid = pd.read_excel(CHEMIN_VALID, dtype=str)
print(f"   Lignes validation : {len(df_valid)}")
print(f"   Colonnes          : {list(df_valid.columns)}")

assert "CAUSE"        in df_valid.columns, "❌ Colonne 'CAUSE' introuvable dans validation2"
assert "Codesite"     in df_valid.columns, "❌ Colonne 'Codesite' introuvable dans validation2"

# Fusion par Codesite
df_valid["Codesite"] = df_valid["Codesite"].str.strip().str.upper()
df["Codesite_upper"] = df["Codesite"].str.strip().str.upper()

df_merge = df_valid.merge(
    df[["Codesite_upper", "CAUSE_PREDITE", "ANALYSE",
        "JUSTIFICATION", "TYPE_CAS"]].rename(
        columns={"Codesite_upper": "Codesite"}),
    on="Codesite",
    how="left"
)

df_merge["CAUSE"]        = df_merge["CAUSE"].fillna("").str.strip()
df_merge["CAUSE_PREDITE"]= df_merge["CAUSE_PREDITE"].fillna("").str.strip()

# Filtrer uniquement :
# 1. Catégories entraînées
# 2. Exclure no root cause
cat_lower = [c.lower() for c in CATEGORIES_ENTRAINEMENT]
df_eval = df_merge[
    df_merge["CAUSE"].str.lower().isin(cat_lower)
].copy()

print(f"   Lignes évaluables (cat. entraînées) : {len(df_eval)}")

# Calculer CORRECT
df_eval["CORRECT"] = (
    df_eval["CAUSE_PREDITE"].str.lower() == df_eval["CAUSE"].str.lower()
)
df_eval["RESULTAT"] = df_eval["CORRECT"].map({True: "✓", False: "✗"})

total   = len(df_eval)
correct = int(df_eval["CORRECT"].sum())
acc     = correct / total * 100 if total > 0 else 0

print(f"\n   Total évalué : {total}")
print(f"   ✓ Corrects   : {correct} ({acc:.1f}%)")
print(f"   ✗ Incorrects : {total - correct} ({100-acc:.1f}%)")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 8 — CALCUL DES MÉTRIQUES DÉTAILLÉES
# ─────────────────────────────────────────────────────────────

erreurs_par_cat = defaultdict(lambda: {"correct": 0, "total": 0, "erreurs": []})
for _, row in df_eval.iterrows():
    cat = row["CAUSE"]
    erreurs_par_cat[cat]["total"] += 1
    if row["CORRECT"]:
        erreurs_par_cat[cat]["correct"] += 1
    else:
        erreurs_par_cat[cat]["erreurs"].append(row["CAUSE_PREDITE"])

# ─────────────────────────────────────────────────────────────
# ÉTAPE 9 — SAUVEGARDE RAPPORT DE PERFORMANCE
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("💾 SAUVEGARDE RAPPORT DE PERFORMANCE")
print("="*60)

with pd.ExcelWriter(RAPPORT_PERF, engine='openpyxl') as writer:

    # Feuille 1 : Résultats détaillés ligne par ligne
    cols_detail = ["Codesite", "CAUSE", "CAUSE_PREDITE", "RESULTAT",
                   "ANALYSE", "JUSTIFICATION", "TYPE_CAS"]
    cols_detail = [c for c in cols_detail if c in df_eval.columns]
    df_eval[cols_detail].to_excel(
        writer, sheet_name="Détails", index=False
    )

    # Feuille 2 : Résumé par catégorie
    resume = []
    for cat in sorted(erreurs_par_cat.keys()):
        s   = erreurs_par_cat[cat]
        a   = s["correct"] / s["total"] * 100 if s["total"] > 0 else 0
        top = Counter(s["erreurs"]).most_common(1)
        resume.append({
            "Catégorie"        : cat,
            "✓ Corrects"       : s["correct"],
            "Total"            : s["total"],
            "✗ Incorrects"     : s["total"] - s["correct"],
            "Accuracy (%)"     : round(a, 1),
            "Statut"           : "✅ Bon" if a >= 80 else ("⚠️ Moyen" if a >= 50 else "❌ Faible"),
            "Erreur principale": top[0][0] if top else "—",
            "Nb erreur princ." : top[0][1] if top else 0,
        })
    df_resume = pd.DataFrame(resume)
    df_resume.to_excel(writer, sheet_name="Par catégorie", index=False)

    # Feuille 3 : Résumé global (pour présentation chef)
    stats_type = df["TYPE_CAS"].value_counts().to_dict()
    global_data = {
        "Indicateur": [
            "Total lignes fichier test",
            "Sans commentaire (no root cause)",
            "Cas normaux traités par le modèle",
            "Cas impactés résolus",
            "Cas impactés non résolus",
            "─────────────────────────",
            "Total évalué (catégories entraînées)",
            "✓ Prédictions correctes",
            "✗ Prédictions incorrectes",
            "Accuracy globale (%)",
            "─────────────────────────",
            "Meilleures catégories (≥80%)",
            "Catégories moyennes (50-79%)",
            "Catégories faibles (<50%)",
            "─────────────────────────",
            "Temps moyen par prédiction (s)",
        ],
        "Valeur": [
            len(df),
            len(idx_no_root),
            len(idx_normaux),
            resolus,
            non_resolus,
            "─────────────────────────",
            total,
            correct,
            total - correct,
            f"{acc:.1f}%",
            "─────────────────────────",
            sum(1 for s in erreurs_par_cat.values()
                if s["total"] > 0 and s["correct"]/s["total"] >= 0.8),
            sum(1 for s in erreurs_par_cat.values()
                if s["total"] > 0 and 0.5 <= s["correct"]/s["total"] < 0.8),
            sum(1 for s in erreurs_par_cat.values()
                if s["total"] > 0 and s["correct"]/s["total"] < 0.5),
            "─────────────────────────",
            round(df.loc[df["TEMPS_S"] > 0, "TEMPS_S"].mean(), 2)
            if "TEMPS_S" in df.columns else "N/A",
        ]
    }
    df_global = pd.DataFrame(global_data)
    df_global.to_excel(writer, sheet_name="Résumé global", index=False)

    # Feuille 4 : Confusions détaillées
    confusion_rows = []
    for cat, s in sorted(erreurs_par_cat.items()):
        if s["erreurs"]:
            for err, nb in Counter(s["erreurs"]).most_common(5):
                confusion_rows.append({
                    "Cause réelle"   : cat,
                    "Cause prédite"  : err,
                    "Nb occurrences" : nb,
                    "Impact (%)"     : round(nb / s["total"] * 100, 1),
                })
    if confusion_rows:
        df_confusion = pd.DataFrame(confusion_rows)
        df_confusion.to_excel(writer, sheet_name="Confusions", index=False)

print(f"   ✅ Rapport sauvegardé : {RAPPORT_PERF}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 10 — AFFICHAGE RÉSUMÉ CONSOLE
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📊 RÉSUMÉ PERFORMANCE — PRÉSENTATION CHEF")
print("="*60)
print(f"\n  Fichier analysé    : test2.xlsx ({len(df)} lignes)")
print(f"  Sans commentaire   : {len(idx_no_root)} → 'no root cause' (exclus évaluation)")
print(f"  Traités modèle     : {len(idx_normaux)} cas normaux")
print(f"  Cas impactés       : {len(idx_impactes)} ({resolus} résolus, {non_resolus} non résolus)")
print(f"\n  ── PERFORMANCE SUR CATÉGORIES ENTRAÎNÉES ──")
print(f"  Total évalué       : {total}")
print(f"  Corrects ✓         : {correct} ({acc:.1f}%)")
print(f"  Incorrects ✗       : {total - correct} ({100-acc:.1f}%)")

print(f"\n  ── PAR CATÉGORIE ──")
print(f"  {'Catégorie':<45} {'OK':>4} {'Tot':>5} {'Acc%':>7}")
print(f"  {'-'*65}")
for cat in sorted(erreurs_par_cat.keys()):
    s   = erreurs_par_cat[cat]
    a   = s["correct"] / s["total"] * 100 if s["total"] > 0 else 0
    flg = "✅" if a >= 80 else ("⚠️" if a >= 50 else "❌")
    print(f"  {flg} {cat:<43} {s['correct']:>4} {s['total']:>5} {a:>6.1f}%")
print(f"  {'-'*65}")
print(f"  {'TOTAL':<45} {correct:>4} {total:>5} {acc:>6.1f}%")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 11 — TÉLÉCHARGEMENT DES FICHIERS
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("⬇️  TÉLÉCHARGEMENT DES FICHIERS")
print("="*60)

print("   Téléchargement : resultats_test2.xlsx ...")
files.download(RESULTATS_TEST)

print("   Téléchargement : rapport_performance.xlsx ...")
files.download(RAPPORT_PERF)

print("\n✅ TOUT EST TERMINÉ")
print("   2 fichiers téléchargés sur ta machine :")
print("   → resultats_test2.xlsx    (toutes les prédictions)")
print("   → rapport_performance.xlsx (rapport pour ton chef)")

Mounted at /content/drive
✅ Configuration OK
   Test    : /content/drive/MyDrive/Lyne/test2.xlsx
   Valid   : /content/drive/MyDrive/Lyne/validation2.xlsx
   Sortie  : /content/drive/MyDrive/Lyne/resultats_test2.xlsx

📂 LECTURE DU FICHIER TEST
   Lignes totales    : 2456
   Colonnes          : ['Nom du Physique', 'Codesite', 'CLUSTER', 'Owner', 'Topology', 'Region', 'Ville', 'Equipementiers', 'Verifications']

🔍 CLASSIFICATION DES LIGNES


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   No root cause : 2006
   Cas normaux   : 348
   Cas impactés  : 102

🤖 TRAITEMENT 1 — CAS NORMAUX
   1/348 | 0.43 ligne/s | ETA 13.5 min
   50/348 | 1.05 ligne/s | ETA 4.7 min
   100/348 | 1.15 ligne/s | ETA 3.6 min
   150/348 | 1.17 ligne/s | ETA 2.8 min
   200/348 | 1.19 ligne/s | ETA 2.1 min
   250/348 | 1.22 ligne/s | ETA 1.3 min
   300/348 | 1.24 ligne/s | ETA 0.6 min
   ✅ 348 cas normaux traités
   📚 348 sites indexés pour résolution des cas impactés

🔗 TRAITEMENT 2 — CAS IMPACTÉS
   ✅ Cas impactés résolus     : 2
   ⚠️  Cas impactés non résolus : 100

💾 SAUVEGARDE FICHIER TEST ENRICHI
   ✅ Sauvegardé : /content/drive/MyDrive/Lyne/resultats_test2.xlsx

📊 ÉVALUATION AVEC validation2.xlsx
   Lignes validation : 347
   Colonnes          : ['Nom du Physique', 'Codesite', 'CLUSTER', 'Owner', 'Topology', 'Region', 'Ville', 'Equipementiers', 'CAUSE', 'Verifications']
   Lignes évaluables (cat. entraînées) : 292

   Total évalué : 292
   ✓ Corrects   : 140 (47.9%)
   ✗ Incorrects : 152

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   Téléchargement : rapport_performance.xlsx ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ TOUT EST TERMINÉ
   2 fichiers téléchargés sur ta machine :
   → resultats_test2.xlsx    (toutes les prédictions)
   → rapport_performance.xlsx (rapport pour ton chef)


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 12 — TEST SUR test-daily.xlsx
# ══════════════════════════════════════════════════════════════
import pandas as pd
import time
import os
from google.colab import files

BASE              = "/content/drive/MyDrive/Lyne"
CHEMIN_TEST_DAILY = f"{BASE}/test-daily.xlsx"
RESULTATS_TEST    = f"{BASE}/resultats_test.xlsx"

# ── Définir CATEGORIES si elle n'existe pas déjà ─────────────
# Remplace cette liste par tes vraies catégories
if 'CATEGORIES' not in dir():
    CATEGORIES = [
        "access-issue", "BSS", "AOF", "EXCLU", "IP VLAN",
        "MPR", "ODU", "PRO-HUAWEI", "PRO-ZTE-NOKIA",
        "MTN", "WAREHOUSE", "ESCO", "IHS", "SPARE-ISSUE"
    ]
    print(f"✅ CATEGORIES défini : {len(CATEGORIES)} catégories")

print("\n" + "="*60)
print("🧪 PARTIE 2 — TEST SUR test-daily.xlsx")
print("="*60)

if not os.path.exists(CHEMIN_TEST_DAILY):
    print(f"❌ Fichier non trouvé : {CHEMIN_TEST_DAILY}")
else:
    xls_test = pd.ExcelFile(CHEMIN_TEST_DAILY, engine='openpyxl')
    tous_res = {}

    for sheet in xls_test.sheet_names:
        df = pd.read_excel(CHEMIN_TEST_DAILY, sheet_name=sheet, engine='openpyxl')
        print(f"\n📋 Sheet '{sheet}' : {len(df)} lignes")

        col_com = next((c for c in df.columns
                        if c.upper() in ["COMMENTAIRES", "COMMENTAIRE",
                                         "VERIFICATION", "COMMENT"]), None)
        col_own = next((c for c in df.columns if c.lower() == "owner"), None)
        col_top = next((c for c in df.columns
                        if "topology" in c.lower() or "topolopgy" in c.lower()), None)
        col_ven = next((c for c in df.columns if "vendor" in c.lower()), None)

        if col_com is None:
            print(f"  ⚠️ Colonne commentaire non trouvée. Colonnes : {list(df.columns)}")
            continue

        df["ANALYSE"]       = ""
        df["JUSTIFICATION"] = ""
        df["CAUSE_PREDITE"] = ""
        df["TEMPS_ANALYSE"] = 0.0

        for idx, row in df.iterrows():
            com = propre(str(row.get(col_com, "")))
            if not com:
                df.at[idx, "ANALYSE"]       = "Commentaire vide"
                df.at[idx, "JUSTIFICATION"] = "Aucun commentaire"
                df.at[idx, "CAUSE_PREDITE"] = "COMMENTAIRE_VIDE"
                continue

            own = propre(str(row.get(col_own, ""))) if col_own else ""
            top = propre(str(row.get(col_top, ""))) if col_top else ""
            ven = propre(str(row.get(col_ven, ""))) if col_ven else ""

            t0 = time.time()
            try:
                analyse, cause_norm, cause_brute = predire(
                    com, own, top, ven, infer_model, infer_tokenizer
                )
            except NameError as e:
                # Variable non définie (ex: CATEGORIES)
                analyse      = com[:100]
                cause_norm   = "ERREUR_CONFIG"
                cause_brute  = f"Variable manquante : {str(e)}"
            except Exception as e:
                analyse      = com[:100]
                cause_norm   = "ERREUR"
                cause_brute  = str(e)[:80]

            df.at[idx, "ANALYSE"]       = analyse
            df.at[idx, "JUSTIFICATION"] = cause_brute
            df.at[idx, "CAUSE_PREDITE"] = cause_norm
            df.at[idx, "TEMPS_ANALYSE"] = round(time.time() - t0, 2)

            if (idx + 1) % 20 == 0:
                print(f"  {idx+1}/{len(df)} traités")

        tous_res[sheet] = df
        print(f"  ✅ Sheet terminée")

    # ── Sauvegarde Excel ─────────────────────────────────────
    with pd.ExcelWriter(RESULTATS_TEST, engine='openpyxl') as writer:
        for sheet, df in tous_res.items():
            cols_prio  = [col_com, "ANALYSE", "JUSTIFICATION",
                          "CAUSE_PREDITE", "TEMPS_ANALYSE"]
            cols_reste = [c for c in df.columns if c not in cols_prio]
            cols_final = [c for c in cols_prio if c in df.columns] + cols_reste
            df[cols_final].to_excel(writer, sheet_name=sheet[:31], index=False)

    print(f"\n✅ Résultats sauvegardés : {RESULTATS_TEST}")

    # ── Statistiques rapides ──────────────────────────────────
    for sheet, df in tous_res.items():
        total   = len(df)
        erreurs = (df["CAUSE_PREDITE"] == "ERREUR").sum()
        vides   = (df["CAUSE_PREDITE"] == "COMMENTAIRE_VIDE").sum()
        ok      = total - erreurs - vides
        print(f"\n📊 Stats '{sheet}' :")
        print(f"   Total     : {total}")
        print(f"   Prédits   : {ok} ({ok/total*100:.1f}%)")
        print(f"   Erreurs   : {erreurs}")
        print(f"   Vides     : {vides}")

    # ── Aperçu ───────────────────────────────────────────────
    print("\n🔍 APERÇU (3 premiers exemples réussis) :")
    for sheet, df in list(tous_res.items())[:1]:
        affiches = 0
        for _, row in df.iterrows():
            cause = str(row.get("CAUSE_PREDITE", ""))
            if cause not in ["COMMENTAIRE_VIDE", "ERREUR", "ERREUR_CONFIG", ""]:
                print(f"\n  Commentaire   : {str(row[col_com])[:100]}...")
                print(f"  Cause prédite : {row['CAUSE_PREDITE']}")
                print(f"  Justification : {row['JUSTIFICATION'][:100]}")
                print(f"  Temps         : {row['TEMPS_ANALYSE']}s")
                affiches += 1
                if affiches >= 3:
                    break

    # ── TÉLÉCHARGEMENT ───────────────────────────────────────
    print("\n" + "="*60)
    print("⬇️  TÉLÉCHARGEMENT DU FICHIER")
    print("="*60)
    print(f"   Fichier : resultats_test.xlsx")
    files.download(RESULTATS_TEST)
    print("✅ Téléchargement lancé")


🧪 PARTIE 2 — TEST SUR test-daily.xlsx

📋 Sheet 'daily break down' : 357 lignes
  20/357 traités
  40/357 traités
  60/357 traités
  80/357 traités
  100/357 traités
  120/357 traités
  140/357 traités
  160/357 traités
  180/357 traités
  200/357 traités
  220/357 traités
  240/357 traités
  260/357 traités
  280/357 traités
  300/357 traités
  320/357 traités
  340/357 traités
  ✅ Sheet terminée

✅ Résultats sauvegardés : /content/drive/MyDrive/Lyne/resultats_test.xlsx

📊 Stats 'daily break down' :
   Total     : 357
   Prédits   : 357 (100.0%)
   Erreurs   : 0
   Vides     : 0

🔍 APERÇU (3 premiers exemples réussis) :

  Commentaire   : (05/06/2026 ticket): INC-20260506-00013544...2026-05-06 22:07:48...Feedback RM:
Dans 10min, ils sero...
  Cause prédite : AKTIVCO Defaut GE & Power Cabinet
  Justification : AKTIVCO Defaut GE & Power Cabinet
  Temps         : 1.52s

  Commentaire   : ( hourly IHS): 06/05/2026 17:28 5:46:45 45 V Grid outage no space // awaiting grid return ,Seraphin .

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Téléchargement lancé


In [ ]:
#══════════════════════════════════════════════════════════════
# CELLULE 13 — ÉVALUATION AVEC valid-daily.xlsx
# ══════════════════════════════════════════════════════════════
import pandas as pd
import time
import os
from google.colab import files
# Redefine configuration variables for standalone execution
BASE               = "/content/drive/MyDrive/Lyne"
CHEMIN_VALID_DAILY = f"{BASE}/valid-daily.xlsx"
RESULTATS_EVAL     = f"{BASE}/evaluation.xlsx"

def propre(v):
    s = str(v).strip()
    return "" if s.lower() in ["nan","n/a","none","","-"] else s

print("\n" + "="*60)
print("📊 PARTIE 3 — ÉVALUATION AVEC valid-daily.xlsx")
print("="*60)

if not os.path.exists(CHEMIN_VALID_DAILY):
    print(f"❌ Fichier non trouvé : {CHEMIN_VALID_DAILY}")
else:
    xls_valid = pd.ExcelFile(CHEMIN_VALID_DAILY, engine='openpyxl')
    resultats_eval = []

    for sheet in xls_valid.sheet_names:
        df_v = pd.read_excel(CHEMIN_VALID_DAILY, sheet_name=sheet, engine='openpyxl')
        print(f"\n📋 Sheet '{sheet}' : {len(df_v)} lignes")

        # Trouver colonnes
        col_com_v   = next((c for c in df_v.columns
                            if c.upper() in ["COMMENTAIRES","COMMENTAIRE",
                                             "VERIFICATION","COMMENT"]), None)
        col_cause_v = next((c for c in df_v.columns
                            if "cause" in c.lower()), None)
        col_own_v   = next((c for c in df_v.columns if c.lower() == "owner"), None)
        col_top_v   = next((c for c in df_v.columns
                            if "topology" in c.lower() or "topolopgy" in c.lower()), None)
        col_ven_v   = next((c for c in df_v.columns if "vendor" in c.lower()), None)

        if col_com_v is None or col_cause_v is None:
            print(f"  ⚠️ Colonnes manquantes. Disponibles : {list(df_v.columns)}")
            continue

        df_v["CAUSE_PREDITE"]  = ""
        df_v["CORRECT"]        = ""
        df_v["ANALYSE"]        = ""

        n_total   = 0
        n_correct = 0
        n_vide    = 0

        for idx, row in df_v.iterrows():
            com         = propre(str(row.get(col_com_v, "")))
            cause_reelle = propre(str(row.get(col_cause_v, "")))

            if not com:
                n_vide += 1
                df_v.at[idx, "CORRECT"] = "VIDE"
                continue
            if not cause_reelle or cause_reelle not in CATEGORIES:
                continue

            own = propre(str(row.get(col_own_v, ""))) if col_own_v else ""
            top = propre(str(row.get(col_top_v, ""))) if col_top_v else ""
            ven = propre(str(row.get(col_ven_v, ""))) if col_ven_v else ""

            try:
                analyse, cause_norm, _ = predire(
                    com, own, top, ven, infer_model, infer_tokenizer
                )
            except Exception as e:
                cause_norm = "ERREUR"
                analyse    = ""

            correct = (cause_norm.lower().strip() == cause_reelle.lower().strip())

            df_v.at[idx, "CAUSE_PREDITE"] = cause_norm
            df_v.at[idx, "ANALYSE"]       = analyse
            df_v.at[idx, "CORRECT"]       = "✅ OUI" if correct else "❌ NON"

            n_total   += 1
            if correct:
                n_correct += 1

            if (idx + 1) % 20 == 0:
                print(f"  {idx+1}/{len(df_v)} évalués — Accuracy en cours : {n_correct}/{n_total}")

        resultats_eval.append({
            "sheet": sheet, "df": df_v,
            "n_total": n_total, "n_correct": n_correct, "n_vide": n_vide,
            "col_com": col_com_v, "col_cause": col_cause_v,
        })

    # ── Sauvegarde évaluation ─────────────────────────────────
    with pd.ExcelWriter(RESULTATS_EVAL, engine='openpyxl') as writer:

        # Feuille résumé global
        lignes_resume = []
        total_global  = 0
        correct_global = 0

        for res in resultats_eval:
            acc = res["n_correct"]/res["n_total"]*100 if res["n_total"] > 0 else 0
            lignes_resume.append({
                "Sheet"         : res["sheet"],
                "Total évalués" : res["n_total"],
                "Corrects ✅"   : res["n_correct"],
                "Incorrects ❌" : res["n_total"] - res["n_correct"],
                "Vides"         : res["n_vide"],
                "Accuracy (%)"  : round(acc, 1),
            })
            total_global   += res["n_total"]
            correct_global += res["n_correct"]

        acc_glob = correct_global / total_global * 100 if total_global > 0 else 0
        lignes_resume.append({
            "Sheet"         : "TOTAL GLOBAL",
            "Total évalués" : total_global,
            "Corrects ✅"   : correct_global,
            "Incorrects ❌" : total_global - correct_global,
            "Vides"         : sum(r["n_vide"] for r in resultats_eval),
            "Accuracy (%)"  : round(acc_glob, 1),
        })

        pd.DataFrame(lignes_resume).to_excel(
            writer, sheet_name="RESUME_GLOBAL", index=False
        )

        # Analyse par catégorie
        for res in resultats_eval:
            df_v = res["df"]
            if "CAUSE_PREDITE" not in df_v.columns:
                continue
            df_filt = df_v[df_v["CORRECT"].isin(["✅ OUI","❌ NON"])].copy()

            par_cat = []
            for cat in CATEGORIES:
                sous = df_filt[df_filt[res["col_cause"]] == cat]
                n    = len(sous)
                if n == 0:
                    continue
                ok  = (sous["CORRECT"] == "✅ OUI").sum()
                par_cat.append({
                    "Catégorie"     : cat,
                    "Total"         : n,
                    "Corrects ✅"   : ok,
                    "Incorrects ❌" : n - ok,
                    "Accuracy (%)"  : round(ok/n*100, 1),
                })
            if par_cat:
                pd.DataFrame(par_cat).to_excel(
                    writer, sheet_name=f"PAR_CAT_{res['sheet'][:20]}", index=False
                )

            # Détail complet
            cols_v = [res["col_com"], res["col_cause"],
                      "CAUSE_PREDITE", "CORRECT", "ANALYSE"]
            cols_v = [c for c in cols_v if c in df_v.columns]
            df_v[cols_v].to_excel(
                writer, sheet_name=f"DETAIL_{res['sheet'][:22]}", index=False
            )

    print(f"\n✅ Évaluation sauvegardée : {RESULTATS_EVAL}")

    # ── Affichage résumé ──────────────────────────────────────
    print("\n" + "="*60)
    print("📊 RÉSULTATS FINAUX")
    print("="*60)
    for r in lignes_resume:
        print(f"\n  Sheet : {r['Sheet']}")
        print(f"  Total évalués  : {r['Total évalués']}")
        print(f"  ✅ Corrects    : {r['Corrects ✅']}")
        print(f"  ❌ Incorrects  : {r['Incorrects ❌']}")
        print(f"  🎯 Accuracy    : {r['Accuracy (%)']} %")

    # Téléchargement
    from google.colab import files
    files.download(RESULTATS_TEST)
    files.download(RESULTATS_EVAL)
    print("\n fichiers telecharges")


📊 PARTIE 3 — ÉVALUATION AVEC valid-daily.xlsx

📋 Sheet 'daily break down' : 450 lignes
  20/450 évalués — Accuracy en cours : 0/19
  40/450 évalués — Accuracy en cours : 0/35
  60/450 évalués — Accuracy en cours : 0/51
  80/450 évalués — Accuracy en cours : 0/69
  100/450 évalués — Accuracy en cours : 0/84
  140/450 évalués — Accuracy en cours : 0/111
  160/450 évalués — Accuracy en cours : 0/123
  200/450 évalués — Accuracy en cours : 0/149
  220/450 évalués — Accuracy en cours : 0/164
  240/450 évalués — Accuracy en cours : 0/183
  260/450 évalués — Accuracy en cours : 0/198
  300/450 évalués — Accuracy en cours : 0/229
  320/450 évalués — Accuracy en cours : 0/244
  340/450 évalués — Accuracy en cours : 0/260
  360/450 évalués — Accuracy en cours : 0/274
  380/450 évalués — Accuracy en cours : 0/281
  400/450 évalués — Accuracy en cours : 0/290
  440/450 évalués — Accuracy en cours : 0/312

✅ Évaluation sauvegardée : /content/drive/MyDrive/Lyne/evaluation.xlsx

📊 RÉSULTATS FINAUX



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 fichiers telecharges


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 13 — ÉVALUATION SUR valid-daily.xlsx - evaluation

# ══════════════════════════════════════════════════════════════

import pandas as pd
import time
import os
from google.colab import files
from collections import defaultdict

BASE               = "/content/drive/MyDrive/Lyne"
CHEMIN_VALID       = f"{BASE}/valid-daily.xlsx"
RESULTATS_VALID    = f"{BASE}/resultats_evaluation.xlsx"

print("="*60)
print("📊 ÉVALUATION DU MODÈLE SUR valid-daily.xlsx")
print("="*60)

if not os.path.exists(CHEMIN_VALID):
    print(f"❌ Fichier non trouvé : {CHEMIN_VALID}")
else:
    xls_valid = pd.ExcelFile(CHEMIN_VALID, engine='openpyxl')
    tous_res  = {}

    # Compteurs globaux
    total_global  = 0
    correct_global = 0
    erreurs_par_cat = defaultdict(lambda: {"correct": 0, "total": 0, "erreurs": []})
    matrice = defaultdict(lambda: defaultdict(int))

    for sheet in xls_valid.sheet_names:
        df = pd.read_excel(CHEMIN_VALID, sheet_name=sheet, engine='openpyxl')
        print(f"\n📋 Sheet '{sheet}' : {len(df)} lignes")

        # Trouver colonnes
        col_com = next((c for c in df.columns
                        if c.upper() in ["COMMENTAIRES","COMMENTAIRE",
                                         "VERIFICATION","COMMENT"]), None)
        col_cause_reelle = next((c for c in df.columns
                                  if c.upper() in ["CAUSE","CAUSE_REELLE",
                                                   "LABEL","CATEGORIE"]), None)
        col_own = next((c for c in df.columns if c.lower() == "owner"), None)
        col_top = next((c for c in df.columns
                        if "topology" in c.lower() or "topolopgy" in c.lower()), None)
        col_ven = next((c for c in df.columns if "vendor" in c.lower()), None)

        if col_com is None:
            print(f"  ⚠️ Colonne commentaire non trouvée : {list(df.columns)}")
            continue
        if col_cause_reelle is None:
            print(f"  ⚠️ Colonne cause réelle non trouvée : {list(df.columns)}")
            print(f"  → Colonnes disponibles : {list(df.columns)}")
            continue

        print(f"  Colonne commentaire  : {col_com}")
        print(f"  Colonne cause réelle : {col_cause_reelle}")

        df["CAUSE_PREDITE"]  = ""
        df["CORRECT"]        = False
        df["TEMPS_ANALYSE"]  = 0.0

        for idx, row in df.iterrows():
            com = propre(str(row.get(col_com, "")))
            cause_reelle = propre(str(row.get(col_cause_reelle, "")))

            if not com or not cause_reelle:
                continue

            own = propre(str(row.get(col_own, ""))) if col_own else ""
            top = propre(str(row.get(col_top, ""))) if col_top else ""
            ven = propre(str(row.get(col_ven, ""))) if col_ven else ""

            t0 = time.time()
            try:
                _, cause_predite, _ = predire(
                    com, own, top, ven, infer_model, infer_tokenizer
                )
            except Exception as e:
                cause_predite = "ERREUR"

            correct = (cause_predite.strip().lower() == cause_reelle.strip().lower())

            df.at[idx, "CAUSE_PREDITE"] = cause_predite
            df.at[idx, "CORRECT"]       = correct
            df.at[idx, "TEMPS_ANALYSE"] = round(time.time() - t0, 2)

            # Compteurs
            total_global  += 1
            if correct:
                correct_global += 1

            erreurs_par_cat[cause_reelle]["total"] += 1
            if correct:
                erreurs_par_cat[cause_reelle]["correct"] += 1
            else:
                erreurs_par_cat[cause_reelle]["erreurs"].append(cause_predite)

            # Matrice de confusion
            matrice[cause_reelle][cause_predite] += 1

            if (idx + 1) % 20 == 0:
                acc_courante = correct_global / total_global * 100
                print(f"  {idx+1}/{len(df)} | Accuracy courante : {acc_courante:.1f}%")

        tous_res[sheet] = df
        print(f"  ✅ Sheet terminée")

    # ── RÉSULTATS GLOBAUX ─────────────────────────────────────
    print("\n" + "="*60)
    print("📊 RÉSULTATS GLOBAUX")
    print("="*60)
    accuracy_globale = correct_global / total_global * 100 if total_global > 0 else 0
    print(f"  Total évalué  : {total_global}")
    print(f"  Corrects      : {correct_global}")
    print(f"  Incorrects    : {total_global - correct_global}")
    print(f"  Accuracy      : {accuracy_globale:.1f}%")

    # ── RÉSULTATS PAR CATÉGORIE ───────────────────────────────
    print("\n" + "="*60)
    print("📊 ACCURACY PAR CATÉGORIE")
    print("="*60)
    print(f"  {'Catégorie':<45} {'Correct':>8} {'Total':>7} {'Acc%':>7}")
    print("-"*72)

    for cat in sorted(erreurs_par_cat.keys()):
        stats = erreurs_par_cat[cat]
        acc   = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
        flag  = "  ⚠️" if acc < 50 else ("  ✅" if acc >= 80 else "")
        print(f"  {cat:<45} {stats['correct']:>8} {stats['total']:>7} {acc:>6.1f}%{flag}")

    print("-"*72)
    print(f"  {'TOTAL':<45} {correct_global:>8} {total_global:>7} {accuracy_globale:>6.1f}%")

    # ── ERREURS FRÉQUENTES PAR CATÉGORIE ─────────────────────
    print("\n" + "="*60)
    print("🔍 CONFUSIONS LES PLUS FRÉQUENTES")
    print("="*60)
    for cat, stats in sorted(erreurs_par_cat.items()):
        if stats["erreurs"]:
            from collections import Counter
            top_erreurs = Counter(stats["erreurs"]).most_common(3)
            print(f"\n  Réelle : {cat}")
            for err, nb in top_erreurs:
                print(f"    → Prédit '{err}' {nb} fois")

    # ── SAUVEGARDE ────────────────────────────────────────────
    print("\n" + "="*60)
    print("💾 SAUVEGARDE")
    print("="*60)

    # Feuille résultats détaillés
    with pd.ExcelWriter(RESULTATS_VALID, engine='openpyxl') as writer:
        for sheet, df in tous_res.items():
            col_cause_reelle = next((c for c in df.columns
                                      if c.upper() in ["CAUSE","CAUSE_REELLE",
                                                       "LABEL","CATEGORIE"]), None)
            cols_prio  = [col_com, col_cause_reelle,
                          "CAUSE_PREDITE", "CORRECT", "TEMPS_ANALYSE"]
            cols_reste = [c for c in df.columns if c not in cols_prio]
            cols_final = [c for c in cols_prio if c and c in df.columns] + cols_reste
            df[cols_final].to_excel(writer, sheet_name=sheet[:31], index=False)

        # Feuille résumé par catégorie
        resume_data = []
        for cat, stats in sorted(erreurs_par_cat.items()):
            acc = stats["correct"] / stats["total"] * 100 if stats["total"] > 0 else 0
            resume_data.append({
                "Catégorie"     : cat,
                "Corrects"      : stats["correct"],
                "Total"         : stats["total"],
                "Accuracy (%)"  : round(acc, 1),
                "Top erreur"    : Counter(stats["erreurs"]).most_common(1)[0][0]
                                  if stats["erreurs"] else "—"
            })
        df_resume = pd.DataFrame(resume_data)
        df_resume.to_excel(writer, sheet_name="Résumé", index=False)

        # Feuille résumé global
        df_global = pd.DataFrame([{
            "Total évalué"   : total_global,
            "Corrects"       : correct_global,
            "Incorrects"     : total_global - correct_global,
            "Accuracy (%)"   : round(accuracy_globale, 1),
        }])
        df_global.to_excel(writer, sheet_name="Global", index=False)

    print(f"✅ Résultats sauvegardés : {RESULTATS_VALID}")

    # ── TÉLÉCHARGEMENT ────────────────────────────────────────
    print("\n⬇️  Téléchargement du fichier d'évaluation...")
    files.download(RESULTATS_VALID)
    print("✅ Téléchargement lancé")

📊 ÉVALUATION DU MODÈLE SUR valid-daily.xlsx

📋 Sheet 'daily break down' : 450 lignes
  Colonne commentaire  : COMMENTAIRE
  Colonne cause réelle : CAUSE
  20/450 | Accuracy courante : 0.0%
  40/450 | Accuracy courante : 0.0%
  60/450 | Accuracy courante : 0.0%
  80/450 | Accuracy courante : 0.0%
  100/450 | Accuracy courante : 0.0%
  120/450 | Accuracy courante : 0.0%
  140/450 | Accuracy courante : 0.0%
  160/450 | Accuracy courante : 0.0%
  180/450 | Accuracy courante : 0.0%
  200/450 | Accuracy courante : 0.0%
  220/450 | Accuracy courante : 0.0%
  240/450 | Accuracy courante : 0.0%
  260/450 | Accuracy courante : 0.0%
  280/450 | Accuracy courante : 0.0%
  300/450 | Accuracy courante : 0.0%
  320/450 | Accuracy courante : 0.0%
  340/450 | Accuracy courante : 0.0%
  360/450 | Accuracy courante : 0.0%
  380/450 | Accuracy courante : 0.0%
  400/450 | Accuracy courante : 0.0%
  420/450 | Accuracy courante : 0.0%
  440/450 | Accuracy courante : 0.0%
  ✅ Sheet terminée

📊 RÉSULTATS GLOBA

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Téléchargement lancé


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 14 — GESTION SITES IMPACTÉS + TEST + ÉVALUATION
# Si le modèle est déjà entraîné, commence directement ici.
# ══════════════════════════════════════════════════════════════

import os, gc, re, time, json, torch, warnings
import pandas as pd
from collections import Counter
from google.colab import drive, files
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
warnings.filterwarnings('ignore')

# ── Monter le Drive si pas encore fait ───────────────────────
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────────
# ⚙️  CHEMINS — modifie si nécessaire
# ─────────────────────────────────────────────────────────────
BASE               = "/content/drive/MyDrive/RAISONNEMENT"
DOSSIER_MERGE      = f"{BASE}/model_final"   # modèle fine-tuné mergé
DOSSIER_LORA       = f"{BASE}/finetuned_lora" # fallback si merge absent
MODEL_NAME         = "Qwen/Qwen2.5-3B-Instruct"

CHEMIN_TEST        = f"{BASE}/test-1-daily.xlsx"
CHEMIN_VALID       = f"{BASE}/valid-daily.xlsx"
RESULTATS_TEST     = f"{BASE}/test1_daily_resultats.xlsx"
RESULTATS_EVAL     = f"{BASE}/test1_daily_evaluation.xlsx"

CATEGORIES = {
    "MPR issue", "BSS Hardware issue",
    "AKTIVCO Defaut GE & Power Cabinet",
    "AKTIVCO Coupure ENEO & Baisse de tension",
    "Coupure ENEO & Baisse de tension",
    "Defaut GE & Power Cabinet",
    "Sharing", "SPARE-ISSUE", "Fiber AOF",
    "Projet OCM (HUAWEI)", "ACCESS-ISSUE",
    "Projet OCM (ZTE, NOKIA, autres projets)",
    "ODU HS", "IP & VLAN", "Warehouse HUAWEI",
}

# ─────────────────────────────────────────────────────────────
# 1. CHARGEMENT DU MODÈLE (sans ré-entraîner)
# ─────────────────────────────────────────────────────────────
print("="*60)
print("🤖 CHARGEMENT DU MODÈLE FINE-TUNÉ")
print("="*60)

gc.collect()
torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(
    DOSSIER_MERGE if os.path.exists(DOSSIER_MERGE) else MODEL_NAME,
    trust_remote_code=True
)
tok.pad_token    = tok.eos_token
tok.padding_side = "right"

if os.path.exists(DOSSIER_MERGE):
    print("⏳ Chargement depuis model_final (merge)...")
    mdl = AutoModelForCausalLM.from_pretrained(
        DOSSIER_MERGE,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("✅ Modèle mergé chargé")
else:
    print("⏳ Merge absent → chargement base + LoRA...")
    q = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=q,
        device_map="auto", trust_remote_code=True,
    )
    mdl = PeftModel.from_pretrained(base, DOSSIER_LORA)
    print("✅ Base + LoRA chargé")

mdl.eval()
print(f"VRAM utilisée : {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# ─────────────────────────────────────────────────────────────
# 2. PROMPTS ET FONCTION D'INFÉRENCE
# ─────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """Tu es un expert en analyse d'incidents réseau télécom au monde.

Tu reçois un commentaire d'incident. Selon le type d'incident, tu peux aussi recevoir :
- Owner : le gestionnaire du site
- Topology : la configuration électrique du site
- Vendors : l'équipementier radio (ZTE, HUAWEI, NOKIA)

Tu dois :
1. Reformuler le commentaire en 2 ou 3 lignes claires (REFORMULATION)
2. Identifier la cause racine parmi ces 15 catégories EXACTES (CAUSE)
3. Justifier pourquoi tu as choisi cette cause (JUSTIFICATION)

Les 15 catégories :
1. MPR issue
2. BSS Hardware issue
3. AKTIVCO Defaut GE & Power Cabinet
4. AKTIVCO Coupure ENEO & Baisse de tension
5. Coupure ENEO & Baisse de tension
6. Defaut GE & Power Cabinet
7. Sharing
8. SPARE-ISSUE
9. Fiber AOF
10. Projet OCM (HUAWEI)
11. ACCESS-ISSUE
12. Projet OCM (ZTE, NOKIA, autres projets)
13. ODU HS
14. IP & VLAN
15. Warehouse HUAWEI

Réponds EXACTEMENT dans ce format :
REFORMULATION: [résumé 2-3 lignes]
CAUSE: [la cause exacte de la liste]
JUSTIFICATION: [explication courte de ton choix]"""

def propre(v):
    s = str(v).strip()
    return "" if s.lower() in ["nan","n/a","none","","-"] else s

def construire_user(com, owner="", topology="", vendors=""):
    u = f"Commentaire : {com.strip()}"
    if propre(owner):    u += f"\nOwner : {propre(owner)}"
    if propre(topology): u += f"\nTopology : {propre(topology)}"
    if propre(vendors):  u += f"\nVendors : {propre(vendors)}"
    return u

def appel_modele(com, owner="", topology="", vendors=""):
    """
    Retourne (reformulation, cause_normalisee, justification).
    """
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": construire_user(com, owner, topology, vendors)},
    ]
    prompt = tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(prompt, return_tensors="pt").to(mdl.device)

    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tok.eos_token_id,
        )
    rep = tok.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Parser la réponse structurée
    reformulation = ""
    cause_brute   = ""
    justification = ""

    for ligne in rep.split("\n"):
        l = ligne.strip()
        if l.upper().startswith("REFORMULATION:"):
            reformulation = l.split(":", 1)[-1].strip()
        elif l.upper().startswith("CAUSE:"):
            cause_brute = l.split(":", 1)[-1].strip()
        elif l.upper().startswith("JUSTIFICATION:"):
            justification = l.split(":", 1)[-1].strip()

    # Fallback si format non respecté
    if not reformulation:
        reformulation = rep[:200].replace("\n"," ")
    if not cause_brute:
        cause_brute = rep[:100]

    # Normaliser la cause vers une catégorie officielle
    cause_norm = cause_brute
    for cat in CATEGORIES:
        if cat.lower() in cause_brute.lower():
            cause_norm = cat
            break

    # Dernier fallback : chercher dans toute la réponse
    if cause_norm not in CATEGORIES:
        for cat in CATEGORIES:
            if cat.lower() in rep.lower():
                cause_norm = cat
                break

    if not justification:
        justification = f"Cause identifiée : {cause_norm}"

    return reformulation, cause_norm, justification

# ─────────────────────────────────────────────────────────────
# 3. DÉTECTION DES SITES IMPACTÉS
# ─────────────────────────────────────────────────────────────

# Patterns pour détecter un site enfant impacté par un parent
PATTERNS_IMPACT = [
    r"impacted\s+by\s+([A-Z]{2,5}_\d{3,4}[^\s,;]*)",
    r"impacted\s+by\s+([A-Za-zÀ-ÿ0-9_\-]+(?:\s[A-Za-zÀ-ÿ0-9_\-]+){0,4})",
    r"incident\s+at\s+([A-Z]{2,5}_\d{3,4}[^\s,;]*)",
    r"incident\s+at\s+([A-Za-zÀ-ÿ0-9_\-]+(?:\s[A-Za-zÀ-ÿ0-9_\-]+){0,4})",
    r"impact[eé]\s+par\s+([A-Z]{2,5}_\d{3,4}[^\s,;]*)",
    r"impact[eé]\s+par\s+([A-Za-zÀ-ÿ0-9_\-]+(?:\s[A-Za-zÀ-ÿ0-9_\-]+){0,4})",
    r"Indisponibilit[eé]\s+des\s+services.*?de\s+([A-Z]{2,5}_\d{3,4}[^\s,;]*)",
    r"Indisponibilit[eé]\s+des\s+services.*?de\s+([A-Za-zÀ-ÿ0-9_\-]+(?:\s[A-Za-zÀ-ÿ0-9_\-]+){0,4})",
    r"autour\s+de\s+([A-Z]{2,5}_\d{3,4}[^\s,;]*)",
]

CODE_PATTERN = re.compile(r'\b[A-Z]{2,5}_\d{3,4}\b')

def detecter_parent(commentaire):
    """
    Détecte si le commentaire signale un site impacté.
    Retourne (is_impacted, code_parent, nom_parent_brut).
    Priorité : code site (ex: LIT_554) > nom textuel.
    """
    if not commentaire:
        return False, None, None

    com_lower = commentaire.lower()

    # Tester chaque pattern
    for pattern in PATTERNS_IMPACT:
        m = re.search(pattern, commentaire, re.IGNORECASE)
        if m:
            texte_parent = m.group(1).strip(" ,;.()")
            # Chercher un code site dans le texte capturé
            codes = CODE_PATTERN.findall(texte_parent)
            if codes:
                return True, codes[0], texte_parent
            # Pas de code → garder le nom brut
            return True, None, texte_parent

    return False, None, None

def normaliser_code(code):
    """Normalise un code site pour la recherche."""
    if not code:
        return ""
    return str(code).strip().upper()

def normaliser_nom(nom):
    """Normalise un nom de site pour la recherche."""
    if not nom:
        return ""
    return str(nom).strip().lower().replace("-","").replace("_","").replace(" ","")

# ─────────────────────────────────────────────────────────────
# 4. LECTURE DES COLONNES DU FICHIER
# ─────────────────────────────────────────────────────────────

def trouver_colonnes(df):
    """Détecte automatiquement les colonnes importantes."""
    cols = {c.strip(): c for c in df.columns}
    col_upper = {c.upper(): c for c in df.columns}

    def chercher(*noms):
        for n in noms:
            if n.upper() in col_upper:
                return col_upper[n.upper()]
            for c in df.columns:
                if n.lower() in c.lower():
                    return c
        return None

    return {
        "code"     : chercher("Codesite","Code_site","CODE"),
        "nom"      : chercher("Nom du Physique","Nom","NOM","SITE"),
        "comment"  : chercher("COMMENTAIRES","COMMENTAIRE","VERIFICATION","COMMENT"),
        "owner"    : chercher("Owner","OWNER"),
        "topology" : chercher("Topology","Topolopgy","TOPOLOGY"),
        "vendors"  : chercher("VENDORS","VENDOR","Vendors"),
        "cause"    : chercher("CAUSE","Cause"),
    }

# ─────────────────────────────────────────────────────────────
# 5. TRAITEMENT PRINCIPAL — test-1-daily.xlsx
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("🧪 TRAITEMENT DE test-1-daily.xlsx")
print("="*60)

if not os.path.exists(CHEMIN_TEST):
    print(f"❌ Fichier non trouvé : {CHEMIN_TEST}")
else:
    xls      = pd.ExcelFile(CHEMIN_TEST, engine='openpyxl')
    sheets   = xls.sheet_names
    all_res  = {}

    for sheet in sheets:
        print(f"\n📋 Sheet '{sheet}'")
        df = pd.read_excel(CHEMIN_TEST, sheet_name=sheet, engine='openpyxl')
        print(f"   {len(df)} lignes")

        C = trouver_colonnes(df)
        col_com = C["comment"]
        col_cod = C["code"]
        col_nom = C["nom"]
        col_own = C["owner"]
        col_top = C["topology"]
        col_ven = C["vendors"]

        if col_com is None:
            print(f"   ⚠️ Colonne commentaire introuvable. Colonnes : {list(df.columns)}")
            all_res[sheet] = df
            continue

        # Initialiser les colonnes résultat
        df["REFORMULATION"] = ""
        df["CAUSE_PREDITE"]  = ""
        df["JUSTIFICATION"]  = ""
        df["IS_IMPACTED"]    = False
        df["PARENT_CODE"]    = ""
        df["PARENT_NOM"]     = ""
        df["TEMPS"]          = 0.0

        # ── ÉTAPE A : Identifier les sites impactés ──────────
        indices_normaux  = []
        indices_impactes = []

        for idx, row in df.iterrows():
            com = propre(str(row.get(col_com, "")))
            is_imp, code_p, nom_p = detecter_parent(com)

            if is_imp:
                df.at[idx, "IS_IMPACTED"] = True
                df.at[idx, "PARENT_CODE"] = propre(code_p or "")
                df.at[idx, "PARENT_NOM"]  = propre(nom_p  or "")
                indices_impactes.append(idx)
            else:
                indices_normaux.append(idx)

        print(f"   Sites normaux   : {len(indices_normaux)}")
        print(f"   Sites impactés  : {len(indices_impactes)}")

        # ── ÉTAPE B : Traiter les sites normaux ──────────────
        # Dictionnaire pour retrouver la cause par code ou nom
        dico_causes = {}   # code_normalisé  → cause
        dico_noms   = {}   # nom_normalisé   → cause

        print(f"\n   🔄 Traitement des sites normaux...")
        for i, idx in enumerate(indices_normaux):
            row = df.loc[idx]
            com = propre(str(row.get(col_com, "")))
            own = propre(str(row.get(col_own, ""))) if col_own else ""
            top = propre(str(row.get(col_top, ""))) if col_top else ""
            ven = propre(str(row.get(col_ven, ""))) if col_ven else ""
            cod = propre(str(row.get(col_cod, ""))) if col_cod else ""
            nom = propre(str(row.get(col_nom, ""))) if col_nom else ""

            if not com:
                df.at[idx, "REFORMULATION"] = "Commentaire vide"
                df.at[idx, "CAUSE_PREDITE"]  = "COMMENTAIRE_VIDE"
                df.at[idx, "JUSTIFICATION"]  = "Aucun commentaire disponible"
                continue

            t0 = time.time()
            try:
                ref, cause, justif = appel_modele(com, own, top, ven)
            except Exception as e:
                ref, cause, justif = com[:150], "ERREUR", str(e)[:80]

            df.at[idx, "REFORMULATION"] = ref
            df.at[idx, "CAUSE_PREDITE"]  = cause
            df.at[idx, "JUSTIFICATION"]  = justif
            df.at[idx, "TEMPS"]          = round(time.time()-t0, 2)

            # Enregistrer dans le dictionnaire
            if cod:
                dico_causes[normaliser_code(cod)] = cause
            if nom:
                dico_noms[normaliser_nom(nom)] = cause

            if (i+1) % 20 == 0:
                print(f"   {i+1}/{len(indices_normaux)} normaux traités")

        print(f"   ✅ Sites normaux terminés")

        # ── ÉTAPE C : Résoudre les sites impactés ────────────
        print(f"\n   🔗 Résolution des sites impactés...")
        n_resolus   = 0
        n_non_resolus = 0

        for idx in indices_impactes:
            row      = df.loc[idx]
            code_p   = propre(str(row["PARENT_CODE"]))
            nom_p    = propre(str(row["PARENT_NOM"]))
            com      = propre(str(row.get(col_com, "")))
            own      = propre(str(row.get(col_own, ""))) if col_own else ""
            top      = propre(str(row.get(col_top, ""))) if col_top else ""
            ven      = propre(str(row.get(col_ven, ""))) if col_ven else ""

            cause_parent = None

            # Chercher par code site parent
            if code_p:
                cause_parent = dico_causes.get(normaliser_code(code_p))

            # Chercher par nom si code non trouvé
            if not cause_parent and nom_p:
                # Essayer différentes variantes du nom
                for variante in [normaliser_nom(nom_p),
                                  normaliser_nom(nom_p.split("_")[0] if "_" in nom_p else nom_p)]:
                    for k, v in dico_noms.items():
                        if variante in k or k in variante:
                            cause_parent = v
                            break
                    if cause_parent:
                        break

            if cause_parent:
                # Hériter de la cause du parent
                df.at[idx, "CAUSE_PREDITE"]  = cause_parent
                df.at[idx, "REFORMULATION"] = (
                    f"Ce site est impacté par le site parent {nom_p or code_p}. "
                    f"La cause racine est héritée du site parent."
                )
                df.at[idx, "JUSTIFICATION"]  = (
                    f"Site enfant impacté par {nom_p or code_p} "
                    f"dont la cause est : {cause_parent}"
                )
                n_resolus += 1
            else:
                # Parent introuvable → le modèle analyse quand même
                t0 = time.time()
                try:
                    ref, cause, justif = appel_modele(com, own, top, ven)
                except Exception as e:
                    ref, cause, justif = com[:150], "ERREUR", str(e)[:80]

                df.at[idx, "REFORMULATION"] = ref
                df.at[idx, "CAUSE_PREDITE"]  = cause
                df.at[idx, "JUSTIFICATION"]  = (
                    f"[Parent {nom_p or code_p} non trouvé — analyse directe] {justif}"
                )
                df.at[idx, "TEMPS"] = round(time.time()-t0, 2)
                n_non_resolus += 1

        print(f"   ✅ Impactés résolus via parent : {n_resolus}")
        print(f"   ⚠️  Impactés traités directement : {n_non_resolus}")

        all_res[sheet] = df

    # ── Sauvegarde fichier test ───────────────────────────────
    print(f"\n💾 Sauvegarde des résultats...")
    with pd.ExcelWriter(RESULTATS_TEST, engine='openpyxl') as writer:
        for sheet, df in all_res.items():
            C = trouver_colonnes(df)

            # Ordre des colonnes : infos site | résultats | reste
            cols_prio = []
            for c in [C["nom"], C["code"], C["comment"],
                      "REFORMULATION", "CAUSE_PREDITE", "JUSTIFICATION",
                      "IS_IMPACTED", "PARENT_CODE", "PARENT_NOM", "TEMPS"]:
                if c and c in df.columns:
                    cols_prio.append(c)

            cols_reste = [c for c in df.columns if c not in cols_prio]
            df[cols_prio + cols_reste].to_excel(
                writer, sheet_name=sheet[:31], index=False
            )

    print(f"✅ Résultats sauvegardés : {RESULTATS_TEST}")

    # ── Aperçu ───────────────────────────────────────────────
    print("\n🔍 APERÇU (3 exemples) :")
    for sheet, df in list(all_res.items())[:1]:
        C   = trouver_colonnes(df)
        n   = 0
        for _, row in df.iterrows():
            if propre(str(row.get("CAUSE_PREDITE",""))) not in ["COMMENTAIRE_VIDE","ERREUR",""]:
                print(f"\n  {'IMPACTÉ' if row['IS_IMPACTED'] else 'NORMAL':8} | "
                      f"Cause : {row['CAUSE_PREDITE']}")
                if row['IS_IMPACTED']:
                    print(f"  Parent : {row['PARENT_NOM'] or row['PARENT_CODE']}")
                print(f"  Reformulation : {str(row['REFORMULATION'])[:120]}")
                print(f"  Justification : {str(row['JUSTIFICATION'])[:120]}")
                n += 1
                if n >= 3: break

# ─────────────────────────────────────────────────────────────
# 6. ÉVALUATION AVEC valid-daily.xlsx
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📊 ÉVALUATION AVEC valid-daily.xlsx")
print("="*60)

if not os.path.exists(CHEMIN_VALID):
    print(f"❌ Fichier non trouvé : {CHEMIN_VALID}")
else:
    xls_v      = pd.ExcelFile(CHEMIN_VALID, engine='openpyxl')
    res_eval   = []

    for sheet in xls_v.sheet_names:
        # Charger les causes validées
        df_v = pd.read_excel(CHEMIN_VALID, sheet_name=sheet, engine='openpyxl')
        Cv   = trouver_colonnes(df_v)

        if Cv["comment"] is None or Cv["cause"] is None:
            print(f"⚠️  Sheet '{sheet}' : colonnes commentaire/cause introuvables")
            continue

        # Charger les prédictions (si le fichier test existe)
        df_pred = all_res.get(sheet)
        if df_pred is None and os.path.exists(RESULTATS_TEST):
            try:
                df_pred = pd.read_excel(
                    RESULTATS_TEST, sheet_name=sheet, engine='openpyxl'
                )
            except Exception:
                df_pred = None

        if df_pred is None:
            print(f"⚠️  Sheet '{sheet}' : prédictions non disponibles")
            continue

        Cp = trouver_colonnes(df_pred)

        print(f"\n📋 Sheet '{sheet}'")

        # Aligner par index (même ordre de lignes)
        n_total    = 0
        n_correct  = 0
        n_vide     = 0
        detail     = []

        min_len = min(len(df_v), len(df_pred))

        for idx in range(min_len):
            cause_val  = propre(str(df_v.iloc[idx].get(Cv["cause"],  "")))
            cause_pred = propre(str(df_pred.iloc[idx].get("CAUSE_PREDITE", "")))
            com        = propre(str(df_v.iloc[idx].get(Cv["comment"], "")))
            ref        = propre(str(df_pred.iloc[idx].get("REFORMULATION", "")))
            justif     = propre(str(df_pred.iloc[idx].get("JUSTIFICATION", "")))

            if not com or cause_pred in ["COMMENTAIRE_VIDE",""]:
                n_vide += 1
                continue
            if not cause_val or cause_val not in CATEGORIES:
                continue

            correct = (cause_pred.lower().strip() == cause_val.lower().strip())
            n_total += 1
            if correct:
                n_correct += 1

            detail.append({
                "Commentaire (extrait)" : com[:120],
                "Reformulation"         : ref[:120],
                "Cause validée"         : cause_val,
                "Cause prédite"         : cause_pred,
                "Justification"         : justif[:120],
                "Résultat"              : "✅ CORRECT" if correct else "❌ INCORRECT",
            })

        acc = n_correct / n_total * 100 if n_total > 0 else 0
        res_eval.append({
            "sheet": sheet, "detail": detail,
            "n_total": n_total, "n_correct": n_correct,
            "n_incorrect": n_total - n_correct,
            "n_vide": n_vide, "accuracy": acc,
        })
        print(f"   ✅ Corrects   : {n_correct}/{n_total}")
        print(f"   ❌ Incorrects : {n_total-n_correct}/{n_total}")
        print(f"   🎯 Accuracy   : {acc:.1f} %")

    # ── Analyse par catégorie ─────────────────────────────────
    print("\n📊 ANALYSE PAR CATÉGORIE :")
    all_details = []
    for r in res_eval:
        all_details += r["detail"]

    df_det = pd.DataFrame(all_details)
    if len(df_det) > 0:
        par_cat = []
        for cat in sorted(CATEGORIES):
            sous = df_det[df_det["Cause validée"] == cat]
            n    = len(sous)
            if n == 0:
                continue
            ok = (sous["Résultat"] == "✅ CORRECT").sum()
            par_cat.append({
                "Catégorie"     : cat,
                "Total"         : n,
                "✅ Corrects"   : ok,
                "❌ Incorrects" : n - ok,
                "Accuracy (%)"  : round(ok/n*100, 1),
            })
            statut = "✅" if ok/n >= 0.7 else "⚠️ "
            print(f"  {statut} {cat}: {ok}/{n} ({ok/n*100:.0f}%)")

    # ── Sauvegarde évaluation ─────────────────────────────────
    with pd.ExcelWriter(RESULTATS_EVAL, engine='openpyxl') as writer:

        # Résumé global
        lignes_res = []
        tot_g = cor_g = 0
        for r in res_eval:
            lignes_res.append({
                "Sheet"         : r["sheet"],
                "Total évalués" : r["n_total"],
                "✅ Corrects"   : r["n_correct"],
                "❌ Incorrects" : r["n_incorrect"],
                "Vides ignorés" : r["n_vide"],
                "Accuracy (%)"  : round(r["accuracy"], 1),
            })
            tot_g += r["n_total"]
            cor_g += r["n_correct"]

        acc_g = cor_g / tot_g * 100 if tot_g > 0 else 0
        lignes_res.append({
            "Sheet"         : "══ TOTAL GLOBAL ══",
            "Total évalués" : tot_g,
            "✅ Corrects"   : cor_g,
            "❌ Incorrects" : tot_g - cor_g,
            "Vides ignorés" : sum(r["n_vide"] for r in res_eval),
            "Accuracy (%)"  : round(acc_g, 1),
        })
        pd.DataFrame(lignes_res).to_excel(
            writer, sheet_name="RESUME_GLOBAL", index=False
        )

        # Par catégorie
        if par_cat:
            pd.DataFrame(par_cat).to_excel(
                writer, sheet_name="PAR_CATEGORIE", index=False
            )

        # Détail ligne par ligne
        if len(df_det) > 0:
            df_det.to_excel(writer, sheet_name="DETAIL_COMPLET", index=False)

            # Uniquement les incorrects
            df_err = df_det[df_det["Résultat"] == "❌ INCORRECT"]
            if len(df_err) > 0:
                df_err.to_excel(writer, sheet_name="INCORRECTS", index=False)

    print(f"\n✅ Évaluation sauvegardée : {RESULTATS_EVAL}")

    # Résumé final
    print("\n" + "="*60)
    print("🏆 RÉSUMÉ FINAL")
    print("="*60)
    print(f"   Total évalués  : {tot_g}")
    print(f"   ✅ Corrects    : {cor_g}")
    print(f"   ❌ Incorrects  : {tot_g - cor_g}")
    print(f"   🎯 Accuracy    : {acc_g:.1f} %")

# ─────────────────────────────────────────────────────────────
# 7. TÉLÉCHARGEMENT DES FICHIERS
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("📥 TÉLÉCHARGEMENT DES FICHIERS")
print("="*60)

for chemin, nom in [
    (RESULTATS_TEST, "test1_daily_resultats.xlsx"),
    (RESULTATS_EVAL, "test1_daily_evaluation.xlsx"),
]:
    if os.path.exists(chemin):
        files.download(chemin)
        print(f"✅ {nom} téléchargé")
    else:
        print(f"⚠️  {nom} non trouvé")

🤖 CHARGEMENT DU MODÈLE FINE-TUNÉ


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

⏳ Merge absent → chargement base + LoRA...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.38 GiB is allocated by PyTorch, and 45.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)